# Foreign Whispers — Competing with the ElevenLabs Dubbing Service

ElevenLabs charges $0.20/minute for automated dubbing. Their service takes a video, transcribes it, translates it, clones the speaker's voice, and returns a dubbed video in the target language. Watch their demo:

**[ElevenLabs Dubbing Demo](https://youtu.be/RKzp4OfCgBA)**

You are going to build the same thing from open-source components. No API keys to a proprietary service. No per-minute billing. The entire pipeline runs on your own GPU server.

The hard problem is not transcription or translation — those are solved. The hard problem is **temporal alignment**: the target-language translation of a 3-second source-language phrase might take 5 seconds to speak. You cannot just stretch the audio — it sounds robotic. You cannot just let segments overlap — the next subtitle arrives while the previous one is still playing. ElevenLabs solves this with proprietary models and massive compute. You will solve it with open-source tools and better engineering.

This notebook is your workbench. By the end, you will have a working pipeline that takes a YouTube video and outputs a dubbed version in the target language with synchronized subtitles — and you will understand exactly where the quality gaps are and how to close them.

You will demonstrate your pipeline using the **Foreign Whispers Dubbing Studio** — a Next.js frontend built specifically for this project. Select a video from the library, run the pipeline, and compare baseline vs. aligned dubbing side by side.

![Foreign Whispers Dubbing Studio](images/frontend-dubbing-studio.png)

Along the way you will gain hands-on experience with production tools that matter beyond this project:

- **FastAPI** — the backend is a layered FastAPI application with Pydantic schemas, dependency injection, and async request handling. You will read, modify, and extend real API endpoints.
- **Logfire** — every pipeline step emits structured traces to Pydantic's observability platform. You will use Logfire to debug timing issues, track per-segment metrics, and compare experiment runs — the same workflow used in production ML systems.
- **Docker Compose** — the full stack runs as four coordinated containers. You will work with bind mounts, GPU passthrough, multi-profile builds, and container-level debugging.

---

## Architecture

The project has three layers:

| Layer | What it is | Where it runs |
|-------|-----------|---------------|
| **GPU services** | Whisper STT (port 8000), XTTS TTS (port 8020) | Dedicated GPU containers |
| **API** | FastAPI orchestrator (port 8080) — proxies to GPU services | CPU container |
| **`foreign_whispers` library** | Alignment logic, metrics, evaluation | Pure Python — no GPU needed |

```
┌────────────────────┐
│  API (CPU :8080)    │  orchestrates the pipeline
└──┬─────────┬───────┘
   │ HTTP    │ HTTP
   ▼         ▼
┌────────┐ ┌────────┐
│ STT    │ │ TTS    │   GPU containers
│ :8000  │ │ :8020  │
└────────┘ └────────┘
```

## How to work with this notebook

The notebook operates in **two phases**:

### Phase 1 — Produce pipeline artifacts (SDK → API → GPU)

Steps P1–P5 and P10–P11 call the API via the `FWClient` SDK to download video, transcribe, translate, synthesize speech, and stitch the final video. These steps populate `pipeline_data/api/` with JSON transcripts, WAV audio, and MP4 video. **You only re-run these when input data changes.**

### Phase 2 — Develop and debug alignment (library, pure Python)

Steps P6–P9 and P12 import directly from the `foreign_whispers` library to compute segment metrics, apply alignment policies, run global optimization, and evaluate results. This is where you iterate:

```
Tweak alignment params in foreign_whispers
  → Re-compute metrics and alignment (pure Python, instant)
  → Inspect predicted stretch factors, overflow, policy distribution
  → When ready to validate with real audio:
      → Call API for TTS + stitch only (seconds, not minutes)
      → Evaluate with actual durations
  → Repeat
```

**Key insight:** download, transcription, and translation artifacts are stable — they don't change when you modify alignment. Only TTS and stitch depend on alignment decisions. So most iteration stays in Phase 2 without touching the GPU services.

### Debugging the `foreign_whispers` library

- **Exploratory analysis:** Load cached JSON transcripts, compute `SegmentMetrics`, visualize distributions, test `decide_action()` thresholds — all in this notebook.
- **Unit tests:** `pytest` tests for alignment, evaluation, and reranking. Debug with `pytest --pdb` for interactive stepping.
- **No GPU needed:** The library is pure Python (stdlib + optional deps). You can debug alignment on a laptop with pre-computed artifacts.

## Pipeline steps

| Step | Phase | Description |
|------|-------|-------------|
| P1 | 1 | Download video + captions |
| P2 | 1 | Speech activity detection (VAD) |
| P3 | 1 | Speaker diarization |
| P4 | 1 | Speech-to-text via Whisper |
| P5 | 1 | Translate source → target language |
| P6 | 2 | Instrument segment timing metrics |
| P7 | 2 | Alignment fallback policy |
| P8 | 2 | Duration-aware translation re-ranking **(student assignment)** |
| P9 | 2 | Global timeline alignment optimizer |
| P10 | 1 | Text-to-speech |
| P11 | 1 | Stitch audio + subtitles into output video |
| P12 | 2 | Evaluation and experiment comparison |

```
YouTube URL
  → [P1]  Download video + captions          ┐
  → [P2]  Speech Activity Detection (VAD)    │ Phase 1
  → [P3]  Speaker Diarization               │ (SDK → API → GPU)
  → [P4]  Whisper Transcription              │
  → [P5]  Translation source → target        ┘
  → [P6]  Segment Metrics Instrumentation    ┐
  → [P7]  Fallback Policy                    │ Phase 2
  → [P8]  Duration-Aware Re-ranking  ★       │ (library)
  → [P9]  Global Alignment Optimizer         ┘
  → [P10] TTS                                ┐ Phase 1
  → [P11] Stitch audio + subtitles           ┘ (SDK → API → GPU)
  → [P12] Evaluation                           Phase 2
```

★ = student assignment — implement `get_shorter_translations()` in `foreign_whispers/reranking.py`

All steps emit **Logfire** spans with structured fields for segment-level debugging.

---
## Requirements

```bash
docker compose --profile nvidia up -d   # start GPU services + API
uv sync                                 # install library deps locally
```

## Setup — SDK client and Logfire tracing

In [1]:
import json
import pathlib

from foreign_whispers import FWClient

API_URL  = "http://localhost:8080"
fw       = FWClient(API_URL)

# Verify API is reachable
print(fw.healthz())
print(f"SDK client ready: {fw}")

{'status': 'ok'}
SDK client ready: FWClient('http://localhost:8080')


In [2]:
import os

try:
    import logfire
    logfire.configure(service_name="foreign-whispers-notebook")
    LOGFIRE_ENABLED = True
    print("Logfire tracing enabled.")
except Exception:
    # Logfire not installed or not authenticated — use no-op shim
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass

    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass

    import types
    logfire = _noop()
    LOGFIRE_ENABLED = False
    print("Logfire not configured — using no-op shim. Run `logfire auth` to enable.")

Logfire tracing enabled.


---
# Phase 1 — Produce pipeline artifacts

These steps call the API via the SDK. All GPU work happens in the STT/TTS containers.
Results are cached on disk — re-running skips already-completed steps.

## P1 — Download Video and Captions

Uses the **Strait of Hormuz** video for development.

In [3]:
VIDEO_URL = "https://www.youtube.com/watch?v=GYQ5yGV_-Oc"

with logfire.span("P1.download"):
    dl = fw.download(VIDEO_URL)

video_id = dl["video_id"]
print(f"Video ID : {video_id}")
print(f"Title    : {dl['title']}")
print(f"Captions : {len(dl['caption_segments'])} segments")
for seg in dl["caption_segments"][:5]:
    print(f"  {seg}")

18:55:30.058 P1.download


Logfire project URL: ]8;id=2760;https://logfire-us.pydantic.dev/pantelis/foreign-whispers\https://logfire-us.pydantic.dev/pantelis/foreign-whispers]8;;\

Video ID : GYQ5yGV_-Oc
Title    : Strait of Hormuz disruption threatens to shake global economy
Captions : 170 segments
  {'start': 2.32, 'end': None, 'text': '60 Minutes overtime.', 'duration': 3.8}
  {'start': 6.48, 'end': None, 'text': "What's the worst case scenario that", 'duration': 3.76}
  {'start': 8.0, 'end': None, 'text': "you're worried about is that it is", 'duration': 4.4}
  {'start': 10.24, 'end': None, 'text': 'closed for weeks and weeks and weeks', 'duration': 5.519}
  {'start': 12.4, 'end': None, 'text': 'here and you start to see the global', 'duration': 5.84}


---
## P2 — Speech Activity Detection (VAD)

VAD runs server-side via the eval endpoint. The silence regions are used by the
global alignment optimizer in P9.

> Requires `silero-vad` installed in the API container. Degrades gracefully if absent.

In [4]:
# VAD is triggered as part of eval_align in P9.
# We run it here to get the silence budget early.
with logfire.span("P2.vad"):
    eval_result = fw.eval_align(video_id)

print(f"Segments from alignment: {eval_result['n_segments']}")
print(f"Gap shifts: {eval_result['n_gap_shifts']}")
print(f"Mild stretches: {eval_result['n_mild_stretches']}")

18:55:31.643 P2.vad
Segments from alignment: 98
Gap shifts: 0
Mild stretches: 37


---
## P3 — Speaker Diarization

Diarization is handled server-side when `HF_TOKEN` is configured in the API.
Results attach speaker labels to the timeline.

> Requires `pyannote.audio` and `FW_HF_TOKEN` set in the API container.

In [5]:
# Diarization runs server-side as part of the eval endpoint.
# Speaker labels are embedded in the alignment results when available.
print("Diarization is handled server-side via the eval endpoint.")
print("Set FW_HF_TOKEN in the API container to enable.")

Diarization is handled server-side via the eval endpoint.
Set FW_HF_TOKEN in the API container to enable.


---
## P4 — Speech-to-Text via Whisper

Whisper creates the segment timestamps that flow through the entire pipeline.
These are the first source of timing and become ground truth for all alignment decisions.

In [6]:
with logfire.span("P4.transcribe", video_id=video_id):
    transcript = fw.transcribe(video_id)

print(f"Language : {transcript['language']}")
print(f"Segments : {len(transcript['segments'])}")
print("\nFirst 3 segments:")
for seg in transcript["segments"][:3]:
    dur = seg["end"] - seg["start"]
    print(f"  [{seg['start']:.1f}s \u2013 {seg['end']:.1f}s ({dur:.1f}s)]  {seg['text'].strip()}")

18:55:31.679 P4.transcribe
Language : en
Segments : 98

First 3 segments:
  [0.0s – 3.6s (3.6s)]  60 minutes over time.
  [3.6s – 9.5s (5.9s)]  What's the worst case scenario that you're worried about?
  [9.5s – 15.6s (6.1s)]  Is that it is closed for weeks and weeks and weeks here and you start to see the global


---
## P5 — Translate Source → Target Language

Baseline translation using **argostranslate** (OpenNMT, offline). Timestamps are preserved; only `text` is rewritten.

> **Key limitation:** the translator has no duration budget parameter. Duration-aware re-ranking is added in P8.

In [7]:
with logfire.span("P5.translate", video_id=video_id):
    translation = fw.translate(video_id)

print(f"Target language: {translation['target_language']}")
print(f"Segments: {len(translation['segments'])}")
print(f"\nEN: {transcript['segments'][0]['text'].strip()}")
print(f"ES: {translation['segments'][0]['text'].strip()}")

18:55:31.690 P5.translate
Target language: es
Segments: 98

EN: 60 minutes over time.
ES: 60 minutos con el tiempo.


---
# Phase 2 — Develop and debug alignment

These steps use the `foreign_whispers` library directly. No GPU needed.
Load the cached transcripts and iterate on alignment parameters.

## P6 — Segment Timing Metrics

Make timing mismatch measurable before trying to fix it. For each segment: source duration, predicted TTS duration (syllable-based heuristic at ~4.5 syl/s for Romance languages), predicted stretch factor, and overflow.

All types and functions are imported from the `foreign_whispers` library.

In [8]:
from foreign_whispers import (
    AlignAction,
    AlignedSegment,
    DurationAwareTTSBackend,
    SegmentMetrics,
    compute_segment_metrics,
    decide_action,
)

# Use the transcripts returned by the API
en_transcript = transcript
es_transcript = translation

with logfire.span("P6.metrics", video_id=video_id):
    all_metrics = compute_segment_metrics(en_transcript, es_transcript)

bad = [m for m in all_metrics if m.predicted_stretch > 1.5]
print(f"Total segments : {len(all_metrics)}")
print(f"Stretch > 1.5x : {len(bad)}  ({100*len(bad)/max(len(all_metrics),1):.0f}%)")
print("\nWorst 5:")
for m in sorted(bad, key=lambda x: -x.predicted_stretch)[:5]:
    print(f"  seg {m.index:3d}  stretch={m.predicted_stretch:.2f}x  overflow={m.overflow_s:.1f}s")
    print(f"    EN: {m.source_text[:55]}")
    print(f"    ES: {m.translated_text[:55]}")

18:55:31.704 P6.metrics
Total segments : 98
Stretch > 1.5x : 12  (12%)

Worst 5:
  seg  93  stretch=2.20x  overflow=2.3s
    EN: But is that what we are witnessing right now?
    ES: ¿Pero eso es lo que estamos presenciando ahora mismo?
  seg  66  stretch=2.05x  overflow=4.3s
    EN: India had been under significant pressure from the U.S.
    ES: India había estado bajo una presión significativa de Es
  seg  16  stretch=1.96x  overflow=4.6s
    EN: For most Gulf states, the only way for their oil to rea
    ES: Para la mayoría de los estados del Golfo, la única mane
  seg  40  stretch=1.91x  overflow=4.4s
    EN: Now everybody wants his oil and Yakhrismas came early f
    ES: Ahora todo el mundo quiere su aceite y Yakhrismas llegó
  seg  53  stretch=1.78x  overflow=3.4s
    EN: The thought that that could be disrupted for a prolonge
    ES: El pensamiento de que eso podría ser perturbado durante


---
## P7 — Alignment Fallback Policy

Replace the implicit "always stretch to fit" with a clear decision per segment:

| Stretch factor | Action |
|---------------|--------|
| ≤ 1.1 | **accept** — no stretch needed |
| 1.1 – 1.4 | **mild_stretch** — pyrubberband within perceptually safe range |
| 1.4 – 1.8 | **gap_shift** — borrow from adjacent silence if available |
| 1.8 – 2.5 | **request_shorter** — escalate to translation re-ranking (P8) |
| > 2.5 | **fail** — log diagnostic, use silence fallback |

In [9]:
action_counts = {a: 0 for a in AlignAction}
for m in all_metrics:
    action = decide_action(m)
    action_counts[action] += 1
    with logfire.span(
        "P7.policy",
        segment_index = m.index,
        stretch       = round(m.predicted_stretch, 3),
        policy_action = action.value,
    ):
        pass

print("Policy distribution:")
for action, count in action_counts.items():
    bar = "\u2588" * count
    print(f"  {action.value:<20} {count:3d}  {bar}")

18:55:31.715 P7.policy
18:55:31.717 P7.policy
18:55:31.718 P7.policy
18:55:31.718 P7.policy
18:55:31.718 P7.policy
18:55:31.718 P7.policy
18:55:31.718 P7.policy
18:55:31.719 P7.policy
18:55:31.719 P7.policy
18:55:31.719 P7.policy
18:55:31.719 P7.policy
18:55:31.719 P7.policy
18:55:31.720 P7.policy
18:55:31.720 P7.policy
18:55:31.720 P7.policy
18:55:31.720 P7.policy
18:55:31.720 P7.policy
18:55:31.721 P7.policy
18:55:31.721 P7.policy
18:55:31.721 P7.policy
18:55:31.721 P7.policy
18:55:31.721 P7.policy
18:55:31.721 P7.policy
18:55:31.722 P7.policy
18:55:31.722 P7.policy
18:55:31.722 P7.policy
18:55:31.722 P7.policy
18:55:31.722 P7.policy
18:55:31.722 P7.policy
18:55:31.723 P7.policy
18:55:31.723 P7.policy
18:55:31.723 P7.policy
18:55:31.723 P7.policy
18:55:31.723 P7.policy
18:55:31.724 P7.policy
18:55:31.724 P7.policy
18:55:31.724 P7.policy
18:55:31.724 P7.policy
18:55:31.724 P7.policy
18:55:31.724 P7.policy
18:55:31.725 P7.policy
18:55:31.725 P7.policy
18:55:31.725 P7.policy
18:55:31.72

---
## P8 — Duration-Aware Translation Re-ranking (Student Assignment)

For segments where the fallback policy decided `REQUEST_SHORTER`, we need shorter target-language translations that fit the time budget.

**This is your assignment.** The function `get_shorter_translations()` in `foreign_whispers/reranking.py` is a stub that returns an empty list. Your task is to implement it.

### Interface

| Parameter | Type | Description |
|-----------|------|-------------|
| `source_text` | `str` | Original source-language segment |
| `baseline_es` | `str` | Baseline target-language translation (from argostranslate) |
| `target_duration_s` | `float` | Time budget in seconds |
| `context_prev` | `str` | Previous segment text (for coherence) |
| `context_next` | `str` | Next segment text (for coherence) |

**Returns:** `list[TranslationCandidate]` sorted shortest first, where each has:
- `text`: the shortened target-language translation
- `char_count`: `len(text)`
- `brevity_rationale`: short note on what was changed

**Duration heuristic:** target-language TTS ≈ 15 chars/second (or ~4.5 syllables/s for Romance languages).

### Approaches to consider

1. **Rule-based** — strip filler words, synonym lookup, contract phrases ("en este momento" → "ahora")
2. **Multiple backends** — paraphrase input, use a second translation model, pick shortest
3. **LLM re-ranking** — use an LLM API to generate condensed alternatives (adds latency/cost)
4. **Hybrid** — rules first, LLM fallback for remaining overflows

In [10]:
from foreign_whispers import get_shorter_translations, TranslationCandidate

# Show what the stub currently does
demo = get_shorter_translations(
    source_text="This is a test sentence.",
    baseline_es="Esta es una oración de prueba.",
    target_duration_s=2.0,
)
print(f"Stub returns: {demo}")
print("\nImplement get_shorter_translations() in foreign_whispers/reranking.py")

Stub returns: []

Implement get_shorter_translations() in foreign_whispers/reranking.py


In [11]:
es_segments = translation["segments"]
need_rerank = [m for m in all_metrics if decide_action(m) == AlignAction.REQUEST_SHORTER]
print(f"Segments needing re-ranking: {len(need_rerank)}")

reranked = 0
for m in need_rerank:
    prev = es_segments[m.index - 1]["text"] if m.index > 0 else ""
    nxt  = es_segments[m.index + 1]["text"] if m.index < len(es_segments) - 1 else ""

    with logfire.span("P8.rerank", segment_index=m.index, stretch=round(m.predicted_stretch, 3)):
        candidates = get_shorter_translations(
            source_text       = m.source_text,
            baseline_es       = m.translated_text,
            target_duration_s = m.source_duration_s,
            context_prev      = prev,
            context_next      = nxt,
        )

    if candidates:
        best = min(candidates, key=lambda c: abs(len(c.text) / 15.0 - m.source_duration_s))
        es_segments[m.index]["text"] = best.text
        reranked += 1
        print(f"  seg {m.index}: {best.text[:50]!r}  ({best.brevity_rationale})")

print(f"\nRe-ranked: {reranked}/{len(need_rerank)} segments")
if reranked == 0:
    print("(stub returns [] — implement get_shorter_translations to see results)")

Segments needing re-ranking: 16
18:55:31.753 P8.rerank
18:55:31.754 P8.rerank
18:55:31.754 P8.rerank
18:55:31.754 P8.rerank
18:55:31.754 P8.rerank
18:55:31.754 P8.rerank
18:55:31.754 P8.rerank
18:55:31.755 P8.rerank
18:55:31.755 P8.rerank
18:55:31.755 P8.rerank
18:55:31.755 P8.rerank
18:55:31.755 P8.rerank
18:55:31.755 P8.rerank
18:55:31.756 P8.rerank
18:55:31.756 P8.rerank
18:55:31.756 P8.rerank

Re-ranked: 0/16 segments
(stub returns [] — implement get_shorter_translations to see results)


---
## P9 — Global Timeline Alignment Optimizer

Stop treating each segment as an isolated timing problem. Build a timeline model
with segments, silence gaps, and cumulative drift, then apply a global pass that can
shift neighboring segments into available silence rather than forcing local stretches.

In [12]:
from foreign_whispers import global_align

# silence_regions would come from VAD (P2) if silero-vad is installed.
# When unavailable, global_align still works but cannot use gap_shift.
silence_regions = []

with logfire.span("P9.global_align", video_id=video_id):
    aligned_segments = global_align(all_metrics, silence_regions)

shifts    = [s for s in aligned_segments if s.action == AlignAction.GAP_SHIFT]
stretches = [s for s in aligned_segments if s.action == AlignAction.MILD_STRETCH]
drift     = aligned_segments[-1].scheduled_end - aligned_segments[-1].original_end if aligned_segments else 0.0

print(f"Total segments : {len(aligned_segments)}")
print(f"Gap shifts     : {len(shifts)}")
print(f"Mild stretches : {len(stretches)}")
print(f"Total drift    : {drift:.2f}s")

18:58:02.430 P9.global_align
Total segments : 98
Gap shifts     : 0
Mild stretches : 37
Total drift    : 0.00s


---
# Back to Phase 1 — Validate with real audio

After tuning alignment parameters above, call the API for TTS and stitching
to hear the result.

## P10 — Text-to-Speech

Synthesizes dubbed audio using **XTTS v2** via the GPU TTS server (port 8020). The `alignment` flag controls whether time-stretching is applied:

| UI setting | `alignment` | What happens |
|------------|------------|--------------|
| **Baseline** | `False` | Raw TTS audio concatenated segment by segment — no duration matching. Fast, but segments drift from the original timeline. |
| **Aligned** | `True` | Each segment is time-stretched via `pyrubberband` to fit the original segment's duration window. Slower, but keeps dubbed audio synchronized with the video. |

The notebook uses `alignment=True` (Aligned mode). To compare, run P10–P11 again with `alignment=False` and listen to the difference in the frontend.

In [ ]:
with logfire.span("P10.tts", video_id=video_id):
    tts_result = fw.tts(video_id, alignment=True)

print(f"Audio path: {tts_result['audio_path']}")
print(f"Config    : {tts_result['config']}")

---
## P11 — Stitch audio and subtitles into output video

Combines the original video with the dubbed WAV using **ffmpeg**.

In [ ]:
with logfire.span("P11.stitch", video_id=video_id):
    stitch_result = fw.stitch(video_id)

print(f"Video path: {stitch_result['video_path']}")

---
# Phase 2 — Evaluate

## P12 — Evaluation and Experiment Comparison

Report alignment quality metrics for the clip.

In [ ]:
from foreign_whispers import clip_evaluation_report, analyze_failures

with logfire.span("P12.evaluation", video_id=video_id):
    report = clip_evaluation_report(all_metrics, aligned_segments)

print("=== Clip Evaluation Report ===")
for k, v in report.items():
    print(f"  {k:<40} {v}")

analysis = analyze_failures(report)
print("\n=== Failure Analysis ===")
print(f"Category    : {analysis.failure_category}")
print(f"Root cause  : {analysis.likely_root_cause}")
print(f"Next change : {analysis.suggested_change}")

---
## Summary

| Step | Phase | Tool | Output |
|------|-------|------|--------|
| P1 Download | 1 | SDK → API → yt-dlp | `raw_video/*.mp4`, `raw_caption/*.txt` |
| P2 VAD | 1 | SDK → API → silero-vad | speech / silence timeline |
| P3 Diarization | 1 | SDK → API → pyannote | speaker-turn timeline |
| P4 Transcribe | 1 | SDK → API → Whisper (GPU) | `raw_transcription/*.json` |
| P5 Translate | 1 | SDK → API → argostranslate | `translated_transcription/*.json` |
| P6 Metrics | 2 | `foreign_whispers` library | duration error, predicted stretch |
| P7 Policy | 2 | `foreign_whispers` library | accept / stretch / shift / retry / fail |
| P8 Re-rank | 2 | `foreign_whispers.reranking` ★ | shorter candidate translations |
| P9 Global align | 2 | `foreign_whispers` library | scheduled segment starts/ends |
| P10 TTS | 1 | SDK → API → XTTS (GPU) | `translated_audio/*.wav` |
| P11 Stitch | 1 | SDK → API → ffmpeg | `translated_video/*.mp4` |
| P12 Eval | 2 | `foreign_whispers` library | clip quality metrics + failure analysis |

★ = student assignment — implement `get_shorter_translations()` in `foreign_whispers/reranking.py`